In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_arch import RRDB_UNet
from realesrgan.archs.rrdb_unet_v3_arch import RRDB_UNet_v3

In [ ]:
pad = 8
device = "cuda"
#device = "cpu"

In [ ]:
model = RRDB_UNet_v3(
    num_in_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=3,
    ae_channel_multipliers = [1,3,9,18,36],
    use_attention=True,
    body_rrdb_blocks = 13,
    inference=True
)

model = RRDB_UNet_v3(
    num_in_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=1,
    ae_channel_multipliers = [1,2,4,8,16],
    use_attention=True,
    body_rrdb_blocks = 6,
    inference=True
)

In [ ]:
print(model)

In [ ]:
img = cv2.imread("tests/data/lq_4/comic.png")
#img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img, (4020*1, 4020*4))
#img = cv2.resize(img, (4020*1, 4020*1))
scale = 4
img = cv2.resize(img, (img.shape[1]*scale,img.shape[0]*scale))
print(img.shape)
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
# Note: pytorch appears to use different gpu code if you exceed some resolution causing it to be very slow.
# compiling with fixed resolution makes it fast again, but it requires bucketing.
# see esrgan_arch_test2

In [ ]:
from realesrgan.real_esrganer_1x import RealESRGANer1x

#esrganer = RealESRGANer1x(None, model, pad, device)
esrganer = RealESRGANer1x("experiments/train_unet_v3_m/models/net_g_latest.pth", model, pad, device) # todo: scale should be passed in here during inference

In [ ]:
out = esrganer.enhance(img)[0]
out.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
res = (out/255 - img/255)*5

# Create a figure with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # Adjust figsize as needed

# Show the original 'res'
axes[0].imshow(res[:, :, ::-1])
axes[0].set_title("res")
axes[0].axis('off')  # Remove axes for better visualization

# Show the negative '-res'
axes[1].imshow(-res[:, :, ::-1])
axes[1].set_title("-res")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
exit()